# Day 09. Exercise 04
# Pipelines and OOP

## 0. Imports

In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from tqdm.notebook import tqdm
from joblib import dump


## 1. Preprocessing pipeline

Create three custom transformers, the first two out of which will be used within a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. `FeatureExtractor()` class:
 - Takes a dataframe with `uid`, `labname`, `numTrials`, `timestamp` from the file [`checker_submits.csv`](https://drive.google.com/file/d/14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw/view?usp=sharing).
 - Extracts `hour` from `timestamp`.
 - Extracts `weekday` from `timestamp` (numbers).
 - Drops the `timestamp` column.
 - Returns the new dataframe.


2. `MyOneHotEncoder()` class:
 - Takes the dataframe from the result of the previous transformation and the name of the target column.
 - Identifies all the categorical features and transforms them with `OneHotEncoder()`. If the target column is categorical too, then the transformation should not apply to it.
 - Drops the initial categorical features.
 - Returns the dataframe with the features and the series with the target column.


3. `TrainValidationTest()` class:
 - Takes `X` and `y`.
 - Returns `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, `stratified`).


In [2]:
class FeatureExtractor:
    def __init__(self):
        pass

    def fit(self, df, y=None):
        return self

    def transform(self, df):
        df = df.copy()
        df = df[['uid', 'labname', 'numTrials', 'timestamp']].copy()
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        df["hour"] = df["timestamp"].dt.hour
        df["dayofweek"] = df["timestamp"].dt.dayofweek
        df = df.drop("timestamp", axis=1)

        return df

In [3]:
class MyOneHotEncoder:
    def __init__(self, target_column):
        self.columns = []
        self.target_column = target_column
    
    def fit(self, df, y=None):
        for column in df.columns:
            if df[column].dtype == "object" and column != self.target_column:
                self.columns.append(column)
        return self

    def transform(self, df):
        df = df.copy()
        encoder = OneHotEncoder(sparse=False)
        encoded_df = encoder.fit_transform(df[self.columns])
        encoded_df = pd.DataFrame(encoded_df, columns=encoder.get_feature_names(self.columns))

        X = pd.concat([encoded_df, df[["numTrials", 'hour']]], axis=1)
        y = df[self.target_column]

        return X, y

In [4]:
class TrainValidationTest:
    def __init__(self):
        pass

    def split(self, X, y):
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)
        X_train, X_valid, y_train, y_valid  = train_test_split(X_train, y_train, test_size=0.2, random_state=21, stratify=y_train)
        
        return X_train, X_valid, X_test, y_train, y_valid, y_test

## 2. Model selection pipeline

`ModelSelection()` class

 - Takes a list of `GridSearchCV` instances and a dict where the keys are the indexes from that list and the values are the names of the models, the example is below in the reverse order (from high-level to low-level perspective):

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), where jobs you can specify by yourself

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Method `choose()` takes `X_train`, `y_train`, `X_valid`, `y_valid` and returns the name of the best classifier among all the models on the validation set
 - Method `best_results()` returns a dataframe with the columns `model`, `params`, `valid_score` where the rows are the best models within each class of models.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.772727
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.801484
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.855288
```

 - When you iterate through the parameters of a model class, print the name of that class and show the progress using `tqdm.notebook`, in the end of the cycle print the best model of that class.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
```

In [5]:
class ModelSelection:
    def __init__(self, grids, grid_dict):
        self.grids = grids
        self.grid_dict = grid_dict
        self.model_params = []

    def choose(self, X_train, y_train, X_valid, y_valid):
        best_valid_score = -1
        for i, gs in enumerate(tqdm(self.grids)):
            model = self.grid_dict[i]
            print(f"Estimator: {model}")

            gs.fit(X_train, y_train)
            best_model = gs.best_estimator_
            best_params = gs.best_params_
            accuracy_train = best_model.score(X_train, y_train)
            accuracy_valid = best_model.score(X_valid, y_valid)

            print(f"Best params: {best_params}\nBest training accuracy: {accuracy_train:.3f}\nValidation set accuracy score for best params: {accuracy_valid:.3f}\n")
            self.model_params.append({"model": model, 
                                      'params': best_params, 
                                      'valid_score': accuracy_valid})
            
            if accuracy_valid > best_valid_score:
                best_valid_score = accuracy_valid
                best_model = model

        print(f"Classifier with best validation set accuracy: {best_model}")
            
    def best_results(self):
        best_models = pd.DataFrame(self.model_params)

        return best_models

## 3. Finalization

`Finalize()` class
 - Takes an estimator.
 - Method `final_score()` takes `X_train`, `y_train`, `X_test`, `y_test` and returns the accuracy of the model as in the example below:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Method `save_model()` takes a path, saves the model to this path and prints that the model was successfully saved.

In [ ]:
class Finalize:
    def __init__(self, model):
        self.model = model
    def final_score(self, X_train, y_train, X_test, y_test):
        self.model.fit(X_train, y_train)
        final_accuracy = self.model.score(X_test, y_test)

        print(f"Accuracy of the final model is {final_accuracy}")

    def save_model(self, path):
        dump(self.model, path, compress=9)
        print(f"{self.model} was successfully saved!")



## 4. Main program

1. Load the data from the file (****name of file****).
2. Create the preprocessing pipeline that consists of two custom transformers: `FeatureExtractor()` and `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
```
3. Use that pipeline and its method `fit_transform()` on the initial dataset.
```
data = preprocessing.fit_transform(df)
```
4. Get `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` using `TrainValidationTest()` and the result of the pipeline.
5. Create an instance of `ModelSelection()`, use the method `choose()` applying it to the models that you want and parameters that you want, get the dataframe of the best results.
6. create an instance of `Finalize()` with your best model, use method `final_score()` and save the model in the format: `name_of_the_model_{accuracy on test dataset}.sav`.

That is it, congrats!

In [7]:
df = pd.read_csv("../data/checker_submits.csv")
df

,uid,labname,numTrials,timestamp
0,user_4,project1,1,2020-04-17 05:19:02.744528
1,user_4,project1,2,2020-04-17 05:22:45.549397
2,user_4,project1,3,2020-04-17 05:34:24.422370
3,user_4,project1,4,2020-04-17 05:43:27.773992
4,user_4,project1,5,2020-04-17 05:46:32.275104
...,...,...,...,...
1681,user_19,laba06s,9,2020-05-21 20:01:48.959966
1682,user_1,laba06s,6,2020-05-21 20:18:54.487900
1683,user_1,laba06s,7,2020-05-21 20:19:06.872761
1684,user_1,laba06s,8,2020-05-21 20:22:41.877806


In [8]:
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), 
                          ('onehot_encoder', MyOneHotEncoder(target_column='dayofweek'))])

X, y  = preprocessing.fit_transform(df)
X

,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,uid_user_16,uid_user_17,...,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1,numTrials,hour
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1,5
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2,5
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3,5
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4,5
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,5,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1681,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,9,20
1682,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,6,20
1683,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,7,20
1684,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,8,20


In [9]:
train_valid_test_split = TrainValidationTest()
X_train, X_valid, X_test, y_train, y_valid, y_test = train_valid_test_split.split(X, y)

In [10]:
svm = SVC()
svm_params =  [{'kernel':('linear', 'rbf', 'sigmoid'), 
                'C':[0.01, 0.1, 1, 1.5, 5, 10], 
                'gamma': ['scale', 'auto'], 
                'class_weight':('balanced', None), 
                'random_state':[21], 
                'probability':[True]}]
gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=-1)

tree = DecisionTreeClassifier()
tree_params = [{'max_depth': [10, 20, 30],
                'min_samples_split': [2, 5, 8, 10],
                'min_samples_leaf': [1, 2, 4],
                'random_state': [21],
                'criterion': ["entropy", "gini"],
                'class_weight': ["balanced", None],
}]
gs_tree = GridSearchCV(estimator=tree, param_grid=tree_params, scoring='accuracy', cv=2, n_jobs=-1)

rf = RandomForestClassifier()
rf_params = {
    'criterion': ["entropy", "gini"],
    'class_weight': ["balanced", None],
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [10, 20, 30],
    'min_samples_split': [2, 5, 8, 10],
    'min_samples_leaf': [1, 2, 4],
    'random_state': [21]
}
gs_rf = GridSearchCV(estimator=rf, param_grid=rf_params, scoring='accuracy', cv=2, n_jobs=-1)

grids = [gs_svm, gs_tree, gs_rf]
grid_dict = {
    0: "SVM",
    1: "Decision Tree", 
    2: "Random Forest"
}


In [11]:
models = ModelSelection(grids, grid_dict)

In [12]:
models.choose(X_train, y_train, X_valid, y_valid)


Estimator: SVM
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.952
Validation set accuracy score for best params: 0.878

Estimator: Decision Tree
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 30, 'min_samples_leaf': 1, 'min_samples_split': 2, 'random_state': 21}
Best training accuracy: 1.000
Validation set accuracy score for best params: 0.867

Estimator: Random Forest
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 30, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100, 'random_state': 21}
Best training accuracy: 1.000
Validation set accuracy score for best params: 0.900


Classifier with best validation set accuracy: Random Forest


In [13]:
data = models.best_results()

data

,model,params,valid_score
0,SVM,"{'C': 10, 'class_weight': None, 'gamma': 'auto...",0.877778
1,Decision Tree,"{'class_weight': 'balanced', 'criterion': 'gin...",0.866667
2,Random Forest,"{'class_weight': 'balanced', 'criterion': 'gin...",0.900000


In [14]:
best_model = RandomForestClassifier(class_weight='balanced', criterion='gini', max_depth=30, min_samples_leaf=1, min_samples_split=2, n_estimators=100, random_state=21)
finalize = Finalize(model=best_model)
finalize.final_score(X_train, y_train, X_test, y_test)

Accuracy of the final model is 0.9201183431952663


In [15]:
finalize.save_model(path="RandomForest_0.9201183431952663.sav")

RandomForestClassifier(class_weight='balanced', max_depth=30, random_state=21) was successfully saved!
